# 3. Supplementary experiments (appendix)

The appendix reports six studies, all produced by one runner: multi-estimand reuse,
direct-DP proxies, the hybrid workload, the Causal-AIM K-sweep, scalability in the
covariate count, and ridge sensitivity. They run at 200 replications (100 for
scalability). Run notebook 0 first so the data is in place.


## Step 1 - run the supplementary studies

`--tasks all` runs all six. Each writes a CSV of raw per-replication rows plus a
`run_manifest.json` into `results_promoted/`.


In [ ]:
import subprocess
cmd = [
    'python', 'run_promoted_experiments.py', '--results-dir', 'results_promoted',
    '--tasks', 'all', '--n-rep', '200', '--n-rep-scalability', '100', '--n-jobs', '20',
    '--use-real-data', '--ihdp-npz-path', 'data/raw/ihdp_npci_1-100.merged.npz', '--twins-csv-path', 'data/raw/twins.csv', '--acic-dir', 'data/raw/acic', '--lalonde-csv-path', 'data/raw/lalonde.csv',
]
subprocess.run(cmd, cwd='..', check=True)


## Step 2 - aggregate into the appendix tables

Each CSV holds raw per-replication rows. Aggregate RMSE as `sqrt(mean(sq_error))` and
coverage as `mean(covered)`, grouped by each study's keys. The two cells below
reproduce the two headline appendix tables.

The multi-estimand table shows that a single DP release supports ATE, ATT, and a
subgroup effect: every cell should reach nominal coverage.


In [ ]:
import pandas as pd, numpy as np

def aggregate(df, keys):
    # RMSE from the squared-error column; coverage from the 0/1 covered indicator.
    return (df.groupby(keys)
              .apply(lambda d: pd.Series({
                  'RMSE': np.sqrt(d['sq_error'].mean()),
                  'coverage': d['covered'].mean()}))
              .round(3))

multi = pd.read_csv('../results_promoted/multi_estimand.csv')
print(aggregate(multi, ['dataset', 'epsilon', 'estimand']))


The direct-DP table compares Causal + NA+MI against output-perturbation proxies of
published direct DP estimators. The comparison is mixed on RMSE by dataset, while
Causal + NA+MI keeps full coverage everywhere.


In [ ]:
direct = pd.read_csv('../results_promoted/direct_dp_baselines.csv')
print(aggregate(direct, ['dataset', 'epsilon', 'method']))


The remaining four CSVs feed the corresponding appendix subsections; aggregate them
the same way:

- `hybrid_workload.csv` - the hybrid causal/generic workload (group by dataset, epsilon, method).
- `aim_operating_point.csv` - the Causal-AIM K-sweep (group by epsilon, K).
- `scalability.csv` - RMSE and synthesis time vs covariate count (group by d).
- `ridge_sensitivity.csv` - sensitivity to the ridge parameter (group by lambda).
